# 01 - Extract Renewable Electricity Generation Units from MaStR

## Overview

This notebook extracts renewable electricity generation units associated with the enercity group from the German Market Master Data Register (MaStR).

The extracted unit inventory forms the basis of the Renewables-Climate Mart, a dataset developed to provide a realistic demonstration environment for a Retrieval-Augmented Generation (RAG)-based Natural Language Interface to Databases (NLIDB).

The MaStR links generation units to their operators through unique market actor identifiers. Therefore, the extraction process first identifies relevant market actors before retrieving their associated renewable generation units.

The resulting unit inventory is used in subsequent steps to estimate daily electricity generation and integrate climate data.

## Data Source

This notebook uses the complete XML export of the German Market Master Data Register (MaStR), published by the Federal Network Agency (Bundesnetzagentur):

https://www.marktstammdatenregister.de/MaStR/Datendownload

All results contained in this repository are based on the MaStR export published on **2026-01-01**.

For this notebook, the following XML datasets are required:

- Market actors (`Marktakteure_*`)
- Biomass generation units (`EinheitenBiomasse`)
- Hydro generation units (`EinheitenWasser`)
- Wind generation units (`EinheitenWind`)
- Solar generation units (`EinheitenSolar_*`)

The files are expected to be organized as follows:

```text
data/
├── raw/
│   └── mastr/
│       ├── marktakteure/
│       │   ├── Marktakteure_1.xml
│       │   ├── Marktakteure_2.xml
│       │   └── ...
│       │
│       └── einheiten/
│           ├── EinheitenBiomasse.xml
│           ├── EinheitenWasser.xml
│           ├── EinheitenWind.xml
│           │
│           └── solar/
│               ├── EinheitenSolar_1.xml
│               ├── EinheitenSolar_2.xml
│               ├── EinheitenSolar_3.xml
│               └── ...
```

The XML files are not included in this repository and must be obtained separately from the official MaStR data export by users wishing to reproduce the extraction process.

In [1]:
import xml.etree.ElementTree as ET
import pandas as pd

from pathlib import Path

In [2]:
DATA_DIR = "../data"

RAW_MASTR_DIR = f"{DATA_DIR}/raw/mastr"

MARKTAKTEURE_DIR = f"{RAW_MASTR_DIR}/marktakteure"
EINHEITEN_DIR = f"{RAW_MASTR_DIR}/einheiten"

SOLAR_DIR = f"{EINHEITEN_DIR}/solar"

## 1. Define Target Company Portfolio

The extraction process begins with a list of companies associated with the enercity group. The portfolio was compiled from the list of shareholdings (*Anteilsbesitz*) published in the enercity Annual Report 2025.

To improve matching accuracy in subsequent steps, company names are normalized by removing common German legal entity suffixes (e.g., *GmbH*, *AG*, *KG*). This reduces the impact of naming variations between the annual report and the MaStR registry.

In addition, several known aliases, project names, and special cases are added manually. These entities are known to be associated with the target portfolio but may not be identifiable through a direct comparison of legal company names alone.

The resulting normalized company list serves as the basis for identifying portfolio-related market actors in the MaStR registry.

In [3]:
# Extracted from the enercity annual report 2025 (https://www.enercity.com/veroeffentlichungen)

enercity_companies = [
    "enercity AG",
    "Danpower GmbH",
    "Elektroanlagenbau Kammeyer GmbH",
    "enercity Contracting GmbH",
    "enercity digital GmbH",
    "enercity Erneuerbare GmbH",
    "enercity Fachpartner-Holding GmbH",
    "enercity Flughafen Netz GmbH",
    "enercity Grid Partner GmbH",
    "enercity Speichervermarktungsgesellschaft mbH",
    "enercitySolution GmbH",
    "KLH Tiefwerk Holding GmbH",
    "The Mother Nature GmbH",
    "enercity Netz GmbH",
    "GKH-Gemeinschaftskraftwerk Hannover GmbH",
    "GHG- Gasspeicher Hannover GmbH",
    "Rockethome Climate Solutions GmbH",
    "GHG- Gasspeicher Hannover GbR",
    #"htp GmbH",
    "Gasnetzgesellschaft Laatzen-Nord mbH",
    "Gasnetzgesellschaft Seelze GmbH & Co. KG",
    "Netzgesellschaft Laatzen GmbH & Co. KG",
    "Netzverwaltungsgesellschaft Laatzen mbH",
    "TRIGIS NET GmbH",
    "Stadtwerke Wunstorf GmbH & Co. KG",
    "Stadtwerke Wunstorf Verwaltungs GmbH",
    "ev-pay GmbH",
    "ROCKETHOME GmbH",
    "Stadtwerke Garbsen GmbH",
    "Autostrom plus GmbH",
    "Thüga Holding GmbH & Co. KGaA",
    "BEH Bioenergie Hannover GmbH",
    "BES Dresden Süd GmbH & Co. KG",
    "BIOREG Energy & Recycling GmbH",
    "Bitterfelder Fernwärme GmbH",
    "Breeze Four GmbH",
    "Carpe Ventos Energie GmbH",
    "Carpe Ventos Wiesmoor I GmbH & Co. KG",
    "Carpe Ventos Wiesmoor III GmbH & Co. KG",
    "Carpe Ventos Wiesmoor IV GmbH & Co. KG",
    "Carpe Ventos Wiesmoor VII GmbH & Co. KG",
    "Carpe Ventos Wiesmoor X GmbH & Co. KG",
    "Carpe Ventos Wiesmoor XI GmbH & Co. KG",
    "Carpe Ventos Wiesmoor XII GmbH & Co. KG",
    "Danpower Biomasse GmbH",
    "Danpower Eesti AS",
    "Danpower Energie Service GmbH",
    "Danpower Grundstücksbesitz Pfaffenhofen GmbH & Co. KG",
    "Danpower Grundstücksverwaltungs GmbH",
    "Danpower Latvia SIA",
    "Danpower New Energy GmbH",
    "Danpower New Energy Holding GmbH",
    "Danpower Pelletproduktion GmbH",
    "Danpower Umwelt GmbH",
    "Danpower Waste to Energy GmbH",
    "eCG Bioenergie GmbH",
    "eCG Verwertungs GmbH",
    "enercity BES Hannover- Misburg GmbH & Co. KG",
    "enercity BES Lehrte- Ahlten 1 GmbH & Co. KG",
    "enercity Contracting Nord GmbH",
    "enercity Engineering GmbH",
    "enercity Erneuerbare Eisenberg- Komplementär GmbH",
    "enercity Erneuerbare Erste Beteiligung GmbH & Co. KG",
    "enercity Erneuerbare Erste Beteiligung Komplementär GmbH",
    "enercity Erneuerbare Erste Windpark Komplementär GmbH",
    "enercity Erneuerbare Holding GmbH & Co. KG",
    "enercity Erneuerbare Nordwest GmbH",
    "enercity Erneuerbare Projekte GmbH & Co. KG",
    "enercity Erneuerbare Tiefenriede GmbH & Co. KG",
    "enercity Erneuerbare Verwaltungs-GmbH",
    "enercity Solarpark Bemerode GmbH & Co. KG",
    "enercity Solarpark Eisenberg GmbH & Co. KG",
    "enercity Solarpark Lehrte-Ahlten GmbH & Co. KG",
    "enercity Solarpark Weener GmbH & Co. KG",
    "enercity Solarpark Zethau GmbH & Co. KG",
    "enercity Tarnow Repowering GmbH & Co. KG",
    "enercity TRICERA Speicher Nossen-Heynitz GmbH & Co. KG",
    "enercity Umspannwerke GmbH",
    "enercity UW Lehrte-Ahlten GmbH & Co. KG",
    "enercity UW Lemwerder GmbH & Co. KG",
    "enercity UW Westerholt & Schweindorf GmbH & Co. KG",
    "enercity Windpark Barnstorf GmbH & Co. KG",
    "enercity Windpark Barnstorf Verwaltungs GmbH & Co. KG",
    "enercity Windpark Beeskow GmbH & Co. KG",
    "enercity Windpark Beeskow II GmbH & Co. KG",
    "enercity Windpark Beuren GmbH",
    "enercity Windpark Bosau GmbH & Co. KG",
    "enercity Windpark Boxberg GmbH & Co. KG",
    "enercity Windpark Esperke GmbH & Co. KG",
    "enercity Windpark Fischbeck GmbH",
    "enercity Windpark Groß Eilstorf GmbH",
    "enercity Windpark Großheide Nord GmbH & Co. KG",
    "enercity Windpark Großheide Süd GmbH & Co. KG",
    "enercity Windpark Jeetze GmbH & Co. KG",
    "enercity Windpark Kabelitz GmbH & Co. KG",
    "enercity Windpark Klettwitz GmbH & Co. KG",
    "enercity Windpark Kostebrau GmbH & Co. KG",
    "enercity Windpark Lauchhammer GmbH & Co. KG",
    "enercity Windpark Lemwerder GmbH",
    "enercity Windpark Lindewitt GmbH",
    "enercity Windpark Mahlwinkel Nord GmbH & Co. KG",
    "enercity Windpark Münstedt II GmbH & Co. KG",
    "enercity Windpark Nenndorf GmbH & Co. KG",
    "enercity Windpark Portfolio GmbH & Co. KG",
    "enercity Windpark Portfolio Horizon GmbH & Co. KG" 
    "Windpark Stemwede GmbH & Co. KG",
    "enercity Windpark Portfolio II GmbH",
    "enercity Windpark Ristedt GmbH & Co. KG",
    "enercity Windpark Schipkau GmbH & Co. KG",
    "enercity Windpark Schipkau Infrastruktur GmbH & Co. KG" 
    "KGE Schipkau-Süd Infrastruktur GmbH & Co. KG",
    "enercity Windpark Schwedt- Heinersdorf GmbH & Co. KG",
    "enercity Windpark Schweindorf GmbH & Co. KG",
    "enercity Windpark Wedemark GmbH & Co. KG",
    "enercity Windpark Westerholt & Schweindorf GmbH & Co. KG",
    "enercity Windpark Wiesmoor I GmbH & Co. KG",
    "enercity Windpark Wölsickendorf II GmbH & Co. KG",
    "enercity Windpark Zölkow- Kladrum GmbH & Co. KG",
    "EWATEC Waste & Energy Management GmbH",
    "Freesen-Wind III GmbH & Co. KG",
    "Freesen-Wind IV GmbH & Co. KG",
    "Freesen-Wind V GmbH & Co. KG",
    "Freesen-Wind VI GmbH & Co. KG",
    "Freesenwind VII GmbH & Co. KG",
    "GWS Schwachstrom- und Sicherungstechnik GmbH",
    "IKW Rüdersdorf GmbH",
    "Ingenieurgesellschaft für Gebäudeautomation mbH",
    "Innerstetal Solar GmbH & Co. KG",
    "KLH Kabel- und Leitungsbau GmbH Hannover",
    "KLH Tiefbau GmbH",
    "LynqTech GmbH",
    "MS-AV Verwaltungs GmbH",
    "Norderland \"Ausblick\" GmbH & Co. KG",
    "Norderland Energie GmbH",
    "Norderland Natur Plan GmbH",
    "Norderland Realisierungs GmbH",
    "PC-Novum GmbH",
    "PME Projektmanagement und Engineering GmbH",
    "Röhler Elektrotechnik GmbH",
    "ROMUS Energie & Innovation GmbH "
    "ÖKW Schleife GmbH",
    "Solarpark Wetzen GmbH ＆ Co. KG",
    "Thermische Abfallbehandlung Lauta GmbH & Co. OHG",
    "UW Eisten GmbH",
    "vigoris Handels GmbH",
    "WEA Aasbruch GmbH & Co. KG",
    "WEA Aasbruch II GmbH & Co. KG",
    "WEA Aasbruch III GmbH & Co. KG",
    "WEA Aasbruch IV GmbH & Co. KG",
    "WEA Sögel Süd I GmbH & Co. KG",
    "WEA Sögel Süd II GmbH & Co. KG",
    "WEA Sögel Süd III GmbH & Co. KG",
    "Windkraftanlage Falkenhagen I GmbH & Co. KG",
    "Windkraftanlage Falkenhagen II GmbH & Co. KG",
    "Windpark Dalena GmbH & Co. KG",
    "Windpark Holtriemer Land GmbH & Co. KG",
    "Windpark Holtriemer Land II GmbH & Co. KG",
    "Windpark Holtriemer Land III GmbH & Co. KG",
    "Windpark Holtriemer Land IV GmbH & Co. KG",
    "Windpark Holtriemer Land V GmbH & Co. KG",
    "Windpark Langewerth I GmbH & Co. KG",
    "Windpark Langewerth II GmbH & Co. KG",
    "Windpark NL Hol GmbH & Co. KG",
    "Windpark NL Hol II GmbH & Co. KG",
    "Windpark NL NOR GmbH & Co. KG",
    "Windpark NL Wies GmbH & Co. KG",
    "Windpark Norderland GmbH & Co. KG Blomberg/Neu schoo I",
    "Windpark Norderland GmbH & Co. KG Blomberg/Neu schoo II",
    "Windpark Norderland GmbH & Co. KG Blomberg/Neu schoo III",
    "Windpark Norderland GmbH & Co. KG Holtriemer Hammrich I",
    "Windpark Norderland GmbH & Co. KG Holtriemer Hammrich II",
    "Windpark Norderland GmbH & Co. KG Holtriemer Hammrich III",
    "Windpark Norderland GmbH & Co. KG Holtriemer Hammrich IV",
    "Windpark Norderland GmbH & Co. KG Holtriemer Hammrich IX",
    "Windpark Norderland GmbH & Co. KG Holtriemer Hammrich V",
    "Windpark Norderland GmbH & Co. KG Holtriemer Hammrich VI",
    "Windpark Norderland GmbH & Co. KG Holtriemer Hammrich VII",
    "Windpark Norderland GmbH & Co. KG Holtriemer Hammrich VIII",
    "Windpark Norderland GmbH & Co. KG Ochtersum I",
    "Windpark Norderland GmbH & Co. KG Ochtersum II",
    "Windpark Norderland GmbH & Co. KG Ochtersum III",
    "Windpark Norderland Verwaltungs- und Beteiligungs GmbH",
    "Windpark Ostermarsch GmbH & Co. KG",
    "Windpark Sögel I GmbH & Co. KG",
    "Windpark Sögel II GmbH & Co. KG",
    "Windpark Sögel III GmbH & Co. KG",
    "Windpark Sögel IV GmbH & Co. KG",
    "Windpark Sögel V GmbH & Co. KG",
    "Windpark Sögel VI GmbH & Co. KG",
    "Windpark Sögel VII GmbH & Co. KG",
    "Windpark Steinweg GmbH & Co. KG",
    "WP Norder Hooker GmbH",
    "WP Norderland Hol I Verwaltungs GmbH",
    "WP Norderland Hol II Verwaltungs GmbH",
    "WP Norderland NOR Verwaltungs GmbH",
    "WP Norderland Wies Verwaltungs GmbH",
    "WSW Energie GmbH",
    "Zacharias Gebäudetechnik GmbH",
    "Luftmeister GmbH",
    "WVZ-Wärmeversorgung Zinnowitz GmbH",
    "Bioenergie Loop GmbH",
    "IEW Biogaspark Wolgast GmbH",
    "Fiba Energieservice GmbH",
    "IEW Innovative Energien Wolgast GmbH",
    "SKW Speicherkraftwerk GmbH",
    "ELW Energieversorgung Leinefelde-Worbis GmbH",
    "EBV Windpark Almstedt-Breinum GmbH & Co. Betriebs-KG",
    "Bioenergie Giesen GmbH",
    "Bioenergie Harber GmbH & Co. KG",
    "Stadtwerk Elsterwerda GmbH",
    "Wärmeversorgung Wolgast GmbH",
    "Hanovolt GmbH",
    "eGP enercity Günter Papenburg GmbH",
    "enercity Windpark Rohne GmbH & Co. KG",
    "enercity Windpark Rohne VerwaltungsGmbH-",
    "Energie-Projektgesellschaft Langenhagen mbH",
    "Energieversorgung Bergen GmbH & Co. KG",
    "Norderland enercity Verwaltungs GmbH",
    "PD energy GmbH",
    "Windpark Bergholz Repowering GmbH & Co. KG",
    "Windpark Münstedt Infra GmbH",
    "Windpark Beeskow Infrastruktur GmbH & Co. KG",
    "Biogas Peine GmbH",
    "Windpark Müden/Aller GmbH",
    "Windpark Jeetze II Infrastruktur GbR",
    "enercity Windpark Schleife GmbH & Co. KG",
    "enercity Windpark Schleife Verwaltungs-GmbH",
    "Energiepark Odenberg GmbH & Co. KG",
    "Energiepark Odenberg Verwaltungs GmbH"
]

len(enercity_companies)

225

In [4]:
german_rechtsformen = [
    "GmbH & Co. Betriebs-KG",
    "GmbH & Co. KGaA",
    "GmbH & Co. KG",
    "GmbH & Co. OHG",
    "SE & Co. KGaA",
    "Stiftung & Co. KG",
    "Ltd. & Co. KG",
    "PartG mbB",
    "GmbH",
    "AG",
    "KG",
    "OHG",
    "GbR",
    "UG",
    "e.V.",
    "e.K.",
    "eG",
    "Ltd.",
    "SE",
    "KGaA",
    "PartG",
    "mbH",
]

In [5]:
# Remove German legal entity suffixes to improve matching against MaStR market actor names.
cleaned_list = []
for text in enercity_companies:
    for sub in german_rechtsformen:
        text = text.replace(sub, "")
    cleaned_list.append(text.strip())

# Add known aliases and project names that are not reliably captured by legal company names alone.
addition = ["Carpe Ventos", "Freesen-Wind", "Freesenwind", "Norderland", "htp GmbH", "Windkraftanlage Falkenhagen"]
cleaned_list = cleaned_list + addition

cleaned_list

['enercity',
 'Danpower',
 'Elektroanlagenbau Kammeyer',
 'enercity Contracting',
 'enercity digital',
 'enercity Erneuerbare',
 'enercity Fachpartner-Holding',
 'enercity Flughafen Netz',
 'enercity Grid Partner',
 'enercity Speichervermarktungsgesellschaft',
 'enercitySolution',
 'KLH Tiefwerk Holding',
 'The Mother Nature',
 'enercity Netz',
 'GKH-Gemeinschaftskraftwerk Hannover',
 'GHG- Gasspeicher Hannover',
 'Rockethome Climate Solutions',
 'GHG- Gasspeicher Hannover',
 'Gasnetzgesellschaft Laatzen-Nord',
 'Gasnetzgesellschaft Seelze',
 'Netzgesellschaft Laatzen',
 'Netzverwaltungsgesellschaft Laatzen',
 'TRIGIS NET',
 'Stadtwerke Wunstorf',
 'Stadtwerke Wunstorf Verwaltungs',
 'ev-pay',
 'ROCKETHOME',
 'Stadtwerke Garbsen',
 'Autostrom plus',
 'Thüga Holding',
 'BEH Bioenergie Hannover',
 'BES Dresden Süd',
 'BIOREG Energy & Recycling',
 'Bitterfelder Fernwärme',
 'Breeze Four',
 'Carpe Ventos Energie',
 'Carpe Ventos Wiesmoor I',
 'Carpe Ventos Wiesmoor III',
 'Carpe Ventos Wie

## 2. Identify Portfolio Market Actors

The normalized company portfolio is matched against the market actor registry contained in the MaStR export.

To efficiently process the large XML files, the registry is parsed incrementally using `iterparse`, avoiding the need to load entire files into memory. For each market actor, the company name is compared against the normalized portfolio list using a case-insensitive substring match.

Whenever a match is identified, key attributes such as the MaStR identifier, company name, legal form, location, and registration details are extracted and stored. The resulting dataset represents the set of market actors potentially associated with the target portfolio and serves as the basis for the subsequent validation step.

In [6]:
def extract_marktakteur_data(xml_file, target_company_names):

    extracted_data = []
    
    # Iterparse to save memory
    context = ET.iterparse(xml_file, events=('end',))
    
    for event, elem in context:
        if elem.tag == 'Marktakteur':
            name_in_xml = elem.findtext('Firmenname')
            
            # Check if the company is part of the predefined list
            if name_in_xml and any(company.lower() in name_in_xml.lower() for company in target_company_names):
                # Pull the data needed
                data = {
                    'Mastr_Nummer': elem.findtext('MastrNummer'),
                    'Firmenname': name_in_xml,
                    'Rechtsform': elem.findtext('Rechtsform'),
                    'Bundesland': elem.findtext('Bundesland'),
                    'PLZ': elem.findtext('Postleitzahl'),
                    'Ort': elem.findtext('Ort'),
                    'Strasse': elem.findtext('Strasse'),
                    'Hausnummer': elem.findtext('Hausnummer'),
                    'Registrierungsdatum': elem.findtext('RegistrierungsdatumMarktakteur'),
                    'Website': elem.findtext('Website'),
                    'HauptwirtdschaftszweigAbteilung': elem.findtext('HauptwirtdschaftszweigAbteilung'),
                    'HauptwirtdschaftszweigGruppe': elem.findtext('HauptwirtdschaftszweigGruppe'),
                    'HauptwirtdschaftszweigAbschnitt': elem.findtext('HauptwirtdschaftszweigAbschnitt')
                }
                extracted_data.append(data)
                print(f"Found: {name_in_xml} -> {data['Mastr_Nummer']}")
            
            # Clear memory
            elem.clear()
            
    return pd.DataFrame(extracted_data)

In [7]:
marktakteure_df = pd.DataFrame()

folder = Path(MARKTAKTEURE_DIR)
num_xml = len(list(folder.rglob("*.xml")))

for i in range(num_xml):
    num = i + 1
    xml_path = MARKTAKTEURE_DIR + '/Marktakteure_' + str(num) + '.xml'
    df_marktakteure = extract_marktakteur_data(xml_path, cleaned_list)
    marktakteure_df = pd.concat([marktakteure_df,df_marktakteure], ignore_index=True)

Found: enercity Netz GmbH -> SNB971746988153
Found: enercity Netz GmbH -> GNB917220120879


Found: Stadtwerke Wunstorf GmbH ＆ Co. KG -> GNB913738110296
Found: Stromversorgung Stadtwerke Garbsen GmbH ＆ Co. -> SNB998265950638


Found: WEA Sögel Süd I GmbH ＆ Co. KG -> ABR923759425026
Found: enercity Contracting GmbH -> ABR942767528161
Found: Thermische Abfallbehandlung Lauta GmbH ＆ Co. oHG -> ABR944873398522
Found: Danpower Energie Service GmbH -> ABR935540117324
Found: Danpower Biomasse Pfaffenhofen GmbH -> ABR923283597176


Found: WSW Energie ＆ Wasser AG -> SEM971474878887
Found: IKW Rüdersdorf GmbH -> ABR969583028477
Found: WSW Energie ＆ Wasser AG -> SOM970076307399
Found: WSW Energie ＆ Wasser AG -> GEM906802231126
Found: Stadtwerke Wunstorf GmbH ＆ Co. KG -> ABR929685968101


Found: WEA Sögel Süd II GmbH ＆ Co. KG -> ABR950774983606
Found: WEA Sögel Süd III GmbH ＆ Co. KG -> ABR947204920836
Found: WSW Energie ＆ Wasser AG -> SOM989528287943
Found: WSW Energie ＆ Wasser AG -> ABR971232363518
Found: WSW Energie ＆ Wasser AG -> ABR943985063915
Found: WSW Energie ＆ Wasser AG -> SOM945493708227
Found: Danpower Pelletproduktion GmbH -> ABR996083385243


...

Found: enercity Windpark Beeskow II GmbH ＆ Co. KG -> ABR944115745732


Found: Windpark Norderland GmbH ＆ Co. KG Blomberg/Neuschoo III -> ABR937414859255


Found: enercity Windpark Zölkow-Kladrum GmbH ＆ Co. KG -> ABR963241667251


Found: enercity Windpark Kabelitz GmbH ＆ Co. KG -> ABR959863776457
Found: enercity Windpark Mahlwinkel Nord GmbH ＆ Co. KG -> ABR959520913027


Found: enercity Windpark Bosau GmbH ＆ Co. KG -> ABR941612291953


In [8]:
len(marktakteure_df)

150

## 3. Validate Market Actor Matches

The matching approach used in the previous step prioritizes completeness over precision and may therefore produce false positives. To improve the quality of the extracted portfolio, the identified market actors are reviewed and validated before proceeding with generation unit extraction.

Potential mismatches are identified by comparing the extracted market actor names with the original company portfolio and manually reviewing unmatched results. Based on this review, a small number of additional matches are retained, while known false positives are excluded from the portfolio.

After validation, the final set of portfolio market actors is exported to `data/interim/portfolio_market_actors.csv`. In addition, a standardized address field is generated to support subsequent data enrichment and geocoding steps.

In [9]:
difference = set(marktakteure_df['Firmenname']) - set(enercity_companies)

# Keep only items where '&' is NOT in the string
difference = {name for name in difference if 'enercity' not in name}

addition_difference = {name for name in difference if any(sub in name for sub in addition)}

difference = {name for name in difference if not any(sub in name for sub in addition)}

addition_difference

{'"Windpark Norderland GmbH ＆ Co. KG, Holtriemer Hammrich IV"',
 'Carpe Ventos Wiesmoor III GmbH ＆ Co. KG',
 'Carpe Ventos Wiesmoor IV GmbH ＆ Co. KG',
 'Carpe Ventos Wiesmoor VII GmbH ＆ Co. KG',
 'Carpe Ventos Wiesmoor X GmbH ＆ Co. KG',
 'Carpe Ventos Wiesmoor XI GmbH ＆ Co. KG',
 'Carpe Ventos Wiesmoor XII GmbH ＆ Co. KG',
 'Carpe Ventos Wiesmoor l GmbH ＆ Co. KG',
 'Freesen-Wind III GmbH ＆ Co. KG',
 'Freesen-Wind IV GmbH ＆ Co. KG',
 'Freesen-Wind V GmbH ＆ Co. KG',
 'Freesen-Wind VI GmbH ＆ Co. KG',
 'Freesenwind VII GmbH ＆ Co. KG',
 'Norderland "Ausblick" GmbH ＆ Co. KG',
 'Norderland Naturstrom GmbH',
 'Windkraftanlage Falkenhagen II GmbH ＆ Co. KG',
 'Windkraftanlage Falkenhagen l GmbH ＆ Co. KG',
 'Windpark Norderland GmbH ＆ Co. Holtriemer Hammrich I',
 'Windpark Norderland GmbH ＆ Co. Holtriemer Hammrich II',
 'Windpark Norderland GmbH ＆ Co. Holtriemer Hammrich III',
 'Windpark Norderland GmbH ＆ Co. KG Blomberg/Neuschoo I',
 'Windpark Norderland GmbH ＆ Co. KG Blomberg/Neuschoo II',
 'Win

In [10]:
# Identify extracted market actors that were not part of the original company list.
difference

{'BES Dresden Süd GmbH ＆ Co. KG',
 'Danpower Biomasse Pfaffenhofen GmbH',
 'EBV Windpark Almstedt-Breinum GmbH ＆ Co. Betriebs- KG',
 'Energie-Projektgesellschaft Langenhagen mit beschränkter Haftung.',
 'FIBA Energieservice GmbH',
 'Innerstetal Solar GmbH ＆ Co. KG',
 'Stadtwerke Garbsen',
 'Stadtwerke Wunstorf GmbH ＆ Co. KG',
 'Stromversorgung Stadtwerke Garbsen GmbH ＆ Co.',
 'Thermische Abfallbehandlung Lauta GmbH ＆ Co. oHG',
 'WEA Aasbruch I GmbH ＆ Co. KG',
 'WEA Aasbruch II GmbH ＆ Co. KG',
 'WEA Aasbruch III GmbH ＆ Co. KG',
 'WEA Aasbruch IV GmbH ＆ Co. KG',
 'WEA Sögel Süd I GmbH ＆ Co. KG',
 'WEA Sögel Süd II GmbH ＆ Co. KG',
 'WEA Sögel Süd III GmbH ＆ Co. KG',
 'WSW Energie ＆ Wasser AG',
 'WSW Energiesysteme GmbH ＆ Co. Windpark Weißenberg KG',
 'Windpark Dalena GmbH ＆ Co. KG',
 'Windpark Holtriemer Land I GmbH ＆ Co. KG',
 'Windpark Holtriemer Land II GmbH ＆ Co. KG',
 'Windpark Holtriemer Land III GmbH ＆ Co. KG',
 'Windpark Holtriemer Land IV GmbH ＆ Co. KG',
 'Windpark Holtriemer Lan

In [11]:
keeplist = ['WEA Aasbruch', 'Sögel', 'BES Dresden Süd','FIBA','Garbsen', 'Wunstorf']

difference = {name for name in difference if not any(sub in name for sub in keeplist)}

difference

{'Danpower Biomasse Pfaffenhofen GmbH',
 'EBV Windpark Almstedt-Breinum GmbH ＆ Co. Betriebs- KG',
 'Energie-Projektgesellschaft Langenhagen mit beschränkter Haftung.',
 'Innerstetal Solar GmbH ＆ Co. KG',
 'Thermische Abfallbehandlung Lauta GmbH ＆ Co. oHG',
 'WSW Energie ＆ Wasser AG',
 'WSW Energiesysteme GmbH ＆ Co. Windpark Weißenberg KG',
 'Windpark Dalena GmbH ＆ Co. KG',
 'Windpark Holtriemer Land I GmbH ＆ Co. KG',
 'Windpark Holtriemer Land II GmbH ＆ Co. KG',
 'Windpark Holtriemer Land III GmbH ＆ Co. KG',
 'Windpark Holtriemer Land IV GmbH ＆ Co. KG',
 'Windpark Holtriemer Land V GmbH ＆ Co. KG',
 'Windpark Langewerth I GmbH ＆ Co. KG',
 'Windpark Langewerth II GmbH ＆ Co. KG',
 'Windpark Ostermarsch GmbH ＆ Co. KG',
 'Windpark Steinweg GmbH ＆ Co. KG',
 'eCG Bioenergie'}

In [12]:
# Exclude known false positives from the extracted market actor set.
removelist = ['Danpower Biomasse Pfaffenhofen GmbH', 'Energie-Projektgesellschaft Langenhagen mit beschränkter Haftung.', 'Stromversorgung Stadtwerke Garbsen GmbH ＆ Co.', 
'WSW Energie ＆ Wasser AG', 'WSW Energiesysteme GmbH ＆ Co. Windpark Weißenberg KG', 'Norderland Naturstrom GmbH']

portfolio_market_actors_df = marktakteure_df[~marktakteure_df['Firmenname'].isin(removelist)].reset_index(drop=True)
portfolio_market_actors_df['Adresse'] = [f"{a} {b} {c} {d}" for a,b,c,d in zip(portfolio_market_actors_df.Strasse.str.replace('ß','ss'),portfolio_market_actors_df.Hausnummer,portfolio_market_actors_df.PLZ,portfolio_market_actors_df.Ort)]

portfolio_market_actors_df.to_csv(f"{DATA_DIR}/interim/portfolio_market_actors.csv", index=False)
portfolio_market_actors_df

,Mastr_Nummer,Firmenname,Rechtsform,Bundesland,PLZ,Ort,Strasse,Hausnummer,Registrierungsdatum,Website,HauptwirtdschaftszweigAbteilung,HauptwirtdschaftszweigGruppe,HauptwirtdschaftszweigAbschnitt,Adresse
0,SNB971746988153,enercity Netz GmbH,429,340,30459,Hannover,Auf der Papenburg,18,2017-05-17T09:45:25.7657488,None,None,None,None,Auf der Papenburg 18 30459 Hannover
1,GNB917220120879,enercity Netz GmbH,429,340,30459,Hannover,Auf der Papenburg,18,2017-05-17T09:47:48.3178321,None,None,None,None,Auf der Papenburg 18 30459 Hannover
2,GNB913738110296,Stadtwerke Wunstorf GmbH ＆ Co. KG,430,340,31515,Wunstorf,An der Nonnenwiese,7,2017-06-21T00:00:00,None,None,None,None,An der Nonnenwiese 7 31515 Wunstorf
3,ABR923759425026,WEA Sögel Süd I GmbH ＆ Co. KG,430,340,26789,Leer,Nessestraße,24,2019-02-06T10:39:05.7724746,None,1215,2686,1165,Nessestrasse 24 26789 Leer
4,ABR942767528161,enercity Contracting GmbH,429,340,30159,Hannover,Osterstraße,63,2019-02-01T08:46:19.3844065,None,1215,2688,1165,Osterstrasse 63 30159 Hannover
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
130,ABR937414859255,Windpark Norderland GmbH ＆ Co. KG Blomberg/Neu...,430,340,26789,Leer,Nessestraße,24,2025-04-24T09:53:55.7954267,None,1215,2686,1165,Nessestrasse 24 26789 Leer
131,ABR963241667251,enercity Windpark Zölkow-Kladrum GmbH ＆ Co. KG,430,340,26789,Leer,Nessestraße,24,2025-07-25T10:39:54.2552521,None,1215,2686,1165,Nessestrasse 24 26789 Leer
132,ABR959863776457,enercity Windpark Kabelitz GmbH ＆ Co. KG,430,340,26789,Leer,Nessestraße,24,2025-09-19T08:39:40.8193029,None,1215,2686,1165,Nessestrasse 24 26789 Leer
133,ABR959520913027,enercity Windpark Mahlwinkel Nord GmbH ＆ Co. KG,430,340,26789,Leer,Nessestraße,24,2025-09-19T08:51:45.2440459,None,1215,2686,1165,Nessestrasse 24 26789 Leer


## 4. Extract and Export Portfolio Generation Units

This step extracts all renewable electricity generation units operated by the validated portfolio market actors.

The MaStR unit files are parsed by energy source. For biomass, wind, and hydro, each source is stored in a single XML file. Solar units are handled separately because the MaStR export splits them across multiple XML files.

For each generation unit, the operator identifier (`AnlagenbetreiberMastrNummer`) is compared against the validated list of portfolio market actors. Matching units are extracted with selected technical, geographic, and administrative attributes, including capacity, commissioning date, operating status, location, coordinates, and energy carrier.

The combined unit inventory is exported to `data/interim/portfolio_generation_units.csv` and used as input for the subsequent asset preparation step.

In [13]:
def extract_plants_by_operators(xml_file, target_abr_list, energy_source):
    # Iterparse to save memory
    context = ET.iterparse(xml_file, events=('end',))
    extracted_data = []

    for event, elem in context:
        if elem.tag == ('Einheit' + energy_source):
            # Check if this plant belongs to one of the companies
            operator_id = elem.findtext('AnlagenbetreiberMastrNummer')
            
            if operator_id in target_abr_list:
                # Pull the data needed
                plant_info = {
                    'EinheitMastrNummer': elem.findtext('EinheitMastrNummer'),
                    'NameStromerzeugungseinheit': elem.findtext('NameStromerzeugungseinheit'),
                    'EinheitBetriebsstatus': elem.findtext('EinheitBetriebsstatus'),
                    'Inbetriebnahmedatum': elem.findtext('Inbetriebnahmedatum'),
                    'Registrierungsdatum': elem.findtext('Registrierungsdatum'),
                    'Energietraeger': elem.findtext('Energietraeger'),
                    'Bruttoleistung': elem.findtext('Bruttoleistung'),
                    'Nettonennleistung': elem.findtext('Nettonennleistung'),
                    'Postleitzahl': elem.findtext('Postleitzahl'),
                    'Ort': elem.findtext('Ort'),
                    'Bundesland': elem.findtext('Bundesland'),
                    'AnlagenbetreiberMastrNummer': operator_id,
                    'DatumLetzteAktualisierung': elem.findtext('DatumLetzteAktualisierung'),
                    'Laengengrad': elem.findtext('Laengengrad'),
                    'Breitengrad': elem.findtext('Breitengrad'),
                }
                extracted_data.append(plant_info)
            
            # Clear memory
            elem.clear()

    return pd.DataFrame(extracted_data)

In [14]:
portfolio_operator_ids = portfolio_market_actors_df['Mastr_Nummer'].tolist()
energy_sources = ['Biomasse','Wind','Wasser'] 
portfolio_generation_units_df = pd.DataFrame()

for e in energy_sources:
    xml_file = f"{EINHEITEN_DIR}/Einheiten{e}.xml"
    df_plants = extract_plants_by_operators(xml_file, portfolio_operator_ids, e)
    portfolio_generation_units_df = pd.concat([portfolio_generation_units_df, df_plants], ignore_index=True)
    print(f"Extracted {len(df_plants)} plants.")

Extracted 136 plants.


Extracted 398 plants.


Extracted 3 plants.


In [15]:
folder = Path(SOLAR_DIR)
num_xml = len(list(folder.rglob("*.xml")))

for i in range(num_xml):
    num = i + 1
    xml_path = SOLAR_DIR + '/EinheitenSolar_' + str(num) + '.xml'
    df_plants = extract_plants_by_operators(xml_path, portfolio_operator_ids, 'Solar')
    portfolio_generation_units_df = pd.concat([portfolio_generation_units_df, df_plants], ignore_index=True)
    print(f"Extracted {len(df_plants)} plants.")

Extracted 1 plants.


Extracted 0 plants.


Extracted 2 plants.


Extracted 0 plants.


Extracted 0 plants.


...

Extracted 0 plants.


Extracted 0 plants.


Extracted 2 plants.


Extracted 12 plants.


Extracted 1 plants.


In [16]:
portfolio_generation_units_df

,EinheitMastrNummer,NameStromerzeugungseinheit,EinheitBetriebsstatus,Inbetriebnahmedatum,Registrierungsdatum,Energietraeger,Bruttoleistung,Nettonennleistung,Postleitzahl,Ort,Bundesland,AnlagenbetreiberMastrNummer,DatumLetzteAktualisierung,Laengengrad,Breitengrad
0,SEE909081639072,BM-BHKW Heizwerk Kirchberg,35,2012-02-02,2019-02-01,2493,1189.000,1189.000,08107,Kirchberg,1413,ABR905920218880,2024-03-21T10:19:22.1843189,12.506653,50.619609
1,SEE909043722354,Heizkraftwerk Osterfeld,35,2009-01-14,2019-02-07,2493,1900.000,1900.000,06721,Osterfeld,1414,ABR996083385243,2019-05-28T08:26:38.0316692,11.934822,51.047045
2,SEE931543799284,"BGA Bitterfeld, Mühlenweg 1c, Anlage 1, BHKW",35,2007-06-19,2019-02-12,2493,625.000,625.000,06749,Bitterfeld-Wolfen,1414,ABR991967582008,2025-12-04T16:08:03.8315274,12.312000,51.630000
3,SEE904234294467,"BGA Kaakstedt, Ort Weiler 11, Anlage 1, BHKW",35,2006-09-05,2019-02-13,2493,526.000,526.000,17268,Gerswalde,1400,ABR991967582008,2024-02-26T16:13:04.0643562,13.779000,53.191000
4,SEE955362298346,"BGA Garlipp, Gewerbegebiet, Anlage 1, BHKW",35,2006-12-11,2019-02-13,2493,536.000,536.000,39628,Bismark,1414,ABR905920218880,2025-12-04T16:21:23.3134157,11.612176,52.639640
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
677,SEE996939393286,"enercity PV MS Otto-Wicht-Str.3, Buxtehude",31,None,2025-11-05,2495,22.950,22.950,21614,Buxtehude,1408,ABR901203303755,2025-11-05T11:03:17.8149575,None,None
678,SEE901943736660,"enercity PV MS Otto-Wicht-Str. 5, Buxtehude",31,None,2025-11-05,2495,29.700,29.000,21614,Buxtehude,1408,ABR901203303755,2025-11-05T11:41:31.5750703,None,None
679,SEE941297015981,WIRUS Rietberg,35,2025-10-09,2025-11-11,2495,1306.820,1100.000,33397,Rietberg,1409,ABR901203303755,2025-11-11T13:08:41.4456866,8.403699,51.737936
680,SEE915406876978,enercity_Lehrte-PV,31,None,2025-12-01,2495,2884.050,2860.000,31275,Lehrte,1408,ABR930870189218,2025-12-01T12:02:22.6372534,10.132603,52.361481


In [17]:
portfolio_generation_units_df.to_csv(f"{DATA_DIR}/interim/portfolio_generation_units.csv", index=False)
print(f"Exported {len(portfolio_generation_units_df):,} generation units.")

Exported 682 generation units.
